# uc_export_utils

**Source:** `00_common/uc_export_utils.py`  
**Purpose:** Databricks notebook auto-generated from framework Python module.


## Section 1: Additional module code and configuration

This cell handles: *Additional module code and configuration*


In [ ]:
"""Unity Catalog-first export helpers for writing table snapshots to cloud storage paths."""

from __future__ import annotations


## Section 2: Define `_is_legacy_mount_path()` helper function

This cell handles: *Define `_is_legacy_mount_path()` helper function*


In [ ]:
def _is_legacy_mount_path(path: str) -> bool:
    normalized = (path or "").strip().lower()
    mnt_prefix = "/" + "mnt" + "/"
    dbfs_mnt_prefix = "dbfs:" + mnt_prefix
    return normalized.startswith(mnt_prefix) or normalized.startswith(dbfs_mnt_prefix)


## Section 3: Define `_require_cloud_path()` helper function

This cell handles: *Define `_require_cloud_path()` helper function*


In [ ]:
def _require_cloud_path(path: str, field_name: str = "path") -> str:
    cleaned = (path or "").strip()
    if not cleaned:
        raise ValueError(f"{field_name} is required")
    if _is_legacy_mount_path(cleaned):
        raise ValueError(
            f"{field_name} must be a cloud URI (for example abfss://... or s3://...), not a legacy mount path: {cleaned}"
        )
    return cleaned


## Section 4: Define `export_table_snapshot()` function with logic for processing

This cell handles: *Define `export_table_snapshot()` function with logic for processing*


In [ ]:
def export_table_snapshot(
    spark,
    table_fqn: str,
    destination_path: str,
    file_format: str = "delta",
    mode: str = "overwrite",
    coalesce_to: int | None = None,
):
    """Export a table snapshot to a cloud storage path.

    Args:
        spark: Active SparkSession.
        table_fqn: Fully qualified source table name catalog.schema.table.
        destination_path: Cloud storage destination URI.
        file_format: delta, parquet, csv, or json.
        mode: overwrite or append.
        coalesce_to: Optional number of output files.
    """

    if not table_fqn or "." not in table_fqn:
        raise ValueError("table_fqn must be a fully qualified table name: catalog.schema.table")

    destination = _require_cloud_path(destination_path, field_name="destination_path")
    fmt = (file_format or "delta").strip().lower()
    write_mode = (mode or "overwrite").strip().lower()

    if fmt not in {"delta", "parquet", "csv", "json"}:
        raise ValueError("file_format must be one of: delta, parquet, csv, json")
    if write_mode not in {"overwrite", "append"}:
        raise ValueError("mode must be overwrite or append")

    df = spark.table(table_fqn)
    if coalesce_to is not None:
        df = df.coalesce(int(coalesce_to))

    writer = df.write.format(fmt).mode(write_mode)
    if fmt == "csv":
        writer = writer.option("header", "true")

    writer.save(destination)
    print(f"Exported {table_fqn} to {destination} as format={fmt}, mode={write_mode}")
